# 키보드로 데모 데이터 수집

주어진 환경에서 데모 데이터를 수집합니다.
이 태스크는 머그컵을 집어서 접시 위에 올려놓는 것입니다. 머그컵이 접시 위에 있고, 그리퍼가 열려 있으며, 엔드이펙터가 머그컵 위에 위치하면 환경이 성공으로 인식합니다.

<img src="./media/teleop.gif" width="480" height="360">

xy 평면 이동은 WASD, z축 이동은 RF, 기울이기는 QE, 나머지 회전은 방향키를 사용합니다.

SPACEBAR로 그리퍼 상태를 토글하고, Z 키를 누르면 현재 에피소드 데이터를 삭제하고 환경을 초기화합니다.

오버레이 이미지의 위치는 다음과 같습니다.
- 우측 상단: 에이전트 뷰
- 우측 하단: 이고센트릭 뷰
- 좌측 상단: 왼쪽 측면 뷰
- 좌측 하단: 탑 뷰


In [8]:
import sys
import random
import numpy as np
import os
from PIL import Image
from mujoco_env.y_env import SimpleEnv
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

In [9]:
# If you want to randomize the object positions, set this to None
# If you fix the seed, the object positions will be the same every time
SEED = 0 
# SEED = None <- Uncomment this line to randomize the object positions

REPO_NAME = 'omy_pnp'
NUM_DEMO = 1 # Number of demonstrations to collect
ROOT = "./demo_data" # The root directory to save the demonstrations

In [ ]:
TASK_NAME = 'Put mug cup on the plate' 
xml_path = './asset/example_scene_y.xml'
# Define the environment
PnPEnv = SimpleEnv(xml_path, seed = SEED, state_type = 'joint_angle')

## 데이터셋 피처 정의 및 데이터셋 생성
데이터셋은 다음과 같이 구성됩니다:
```
fps = 20,
features={
    "observation.image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.wrist_image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channel"],
    },
    "observation.state": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["state"], # x, y, z, roll, pitch, yaw
    },
    "action": {
        "dtype": "float32",
        "shape": (7,),
        "names": ["action"], # 6 joint angles and 1 gripper
    },
    "obj_init": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["obj_init"], # just the initial position of the object. Not used in training.
    },
},
```


이 설정은 `./demo_data` 폴더에 데이터셋을 생성하며, 구조는 다음과 같습니다.

```
.
├── data
│   ├── chunk-000
│   │   ├── episode_000000.parquet
│   │   ├── ...
├── meta
│   ├── episodes.jsonl
│   ├── info.json
│   ├── stats.json
│   ├── tasks.jsonl
```


In [11]:
create_new = True
if os.path.exists(ROOT):
    print(f"Directory {ROOT} already exists.")
    ans = input("Do you want to delete it? (y/n) ")
    if ans == 'y':
        import shutil
        shutil.rmtree(ROOT)
    else:
        create_new = False


if create_new:
    dataset = LeRobotDataset.create(
                repo_id=REPO_NAME,
                root = ROOT, 
                robot_type="omy",
                fps=20, # 20 frames per second
                features={
                    "observation.image": {
                        "dtype": "image",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channels"],
                    },
                    "observation.wrist_image": {
                        "dtype": "image",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channel"],
                    },
                    "observation.state": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["state"], # x, y, z, roll, pitch, yaw
                    },
                    "action": {
                        "dtype": "float32",
                        "shape": (7,),
                        "names": ["action"], # 6 joint angles and 1 gripper
                    },
                    "obj_init": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["obj_init"], # just the initial position of the object. Not used in training.
                    },
                },
                image_writer_threads=10,
                image_writer_processes=5,
        )
else:
    print("Load from previous dataset")
    dataset = LeRobotDataset(REPO_NAME, root=ROOT)

Directory ./demo_data already exists.


## 키보드 조작법
키보드로 로봇을 텔레오퍼레이션하여 데이터셋을 수집할 수 있습니다.
```
---------     -----------------------
   w       ->          후진
s  a  d          좌측    전진    우측
---------      -----------------------
x, y 평면 이동

---------
R: 위로 이동
F: 아래로 이동
---------
z축 이동

---------
Q: 왼쪽으로 기울이기
E: 오른쪽으로 기울이기
UP: 위를 보기
Down: 아래를 보기
Right: 오른쪽으로 회전
Left: 왼쪽으로 회전
---------
회전 조작

---------
SPACEBAR: 그리퍼 열기/닫기 토글
--------

---------
z: 초기화
--------
```
환경을 초기화하면 현재 데모의 캐시 데이터가 삭제되고 수집을 다시 시작합니다.


### 이제 로봇을 텔레오퍼레이션하여 데이터를 수집해봅시다!

**성공 신호를 받으려면 그리퍼를 열고 머그컵 위로 올라가야 합니다!**


In [ ]:
PnPEnv.init_viewer()

action = np.zeros(7)
episode_id = 0
record_flag = False # Start recording when the robot starts moving
while PnPEnv.env.is_viewer_alive() and episode_id < NUM_DEMO:
    PnPEnv.step_env()
    if PnPEnv.env.loop_every(HZ=20):
        # check if the episode is done
        done = PnPEnv.check_success()
        if done: 
            # Save the episode data and reset the environment
            dataset.save_episode()
            PnPEnv.reset(seed = SEED)
            episode_id += 1
        # Teleoperate the robot and get delta end-effector pose with gripper
        action, reset  = PnPEnv.teleop_robot()
        if not record_flag and sum(action) != 0:
            record_flag = True
            print("Start recording")
        if reset:
            # Reset the environment and clear the episode buffer
            # This can be done by pressing 'z' key
            PnPEnv.reset(seed=SEED)
            # PnPEnv.reset()
            dataset.clear_episode_buffer()
            record_flag = False
        # Step the environment
        # Get the end-effector pose and images
        ee_pose = PnPEnv.get_ee_pose()
        agent_image,wrist_image = PnPEnv.grab_image()
        # # resize to 256x256
        agent_image = Image.fromarray(agent_image)
        wrist_image = Image.fromarray(wrist_image)
        agent_image = agent_image.resize((256, 256))
        wrist_image = wrist_image.resize((256, 256))
        agent_image = np.array(agent_image)
        wrist_image = np.array(wrist_image)
        joint_q = PnPEnv.step(action)
        if record_flag:
            # Add the frame to the dataset
            dataset.add_frame( {
                    "observation.image": agent_image,
                    "observation.wrist_image": wrist_image,
                    "observation.state": ee_pose, 
                    "action": joint_q,
                    "obj_init": PnPEnv.obj_init_pose,
                    # "task": TASK_NAME,
                }, task = TASK_NAME
            )
        PnPEnv.render(teleop=True)

In [13]:
PnPEnv.env.close_viewer()

In [14]:
# Clean up the images folder
import shutil
shutil.rmtree(dataset.root / 'images')